# 10 — Verification of reported metrics

This notebook re-derives every headline number quoted in the dissertation directly from the
frozen prediction arrays and result CSVs, so that each reported figure is traceable to source.
Cell references below use the `In [n]` execution numbers shown beside each cell.

| Cell | Verifies | Dissertation reference |
|---|---|---|
| `In [5]`, `In [8]` | Test-set metrics for all four models; accuracy of the GAT-only run recovered from its logged sensitivity and precision | Table 4.1 |
| `In [9]`, `In [10]` | Positive/negative score separation, and the metformin score gap | Section 5.1 |
| `In [11]`, `In [12]` | Within-drug AUROC, size-weighted mean, correlation of positive rate with AUROC | Section 5.1, Appendix A |
| `In [7]` | Concentration of SHAP attribution across the 768 BioBERT dimensions | Section 5.1 |
| `In [17]`-`In [19]` | Count-only AUROC, and equivalence of the XGBoost baseline to the rule *a > 0* | Section 4.2 |
| `In [20]`-`In [25]` | Train/test drug-ADR pair overlap; stratified AUROC for previously observed and novel pairs; writes `results/pair_overlap_audit.csv` | Table 4.2 |
| `In [26]` | SIDER external validation: precision, recall, and sensitivity to the matching strategy | Section 5.4 |

Paths assume the project directory `/content/drive/MyDrive/ADR_Project` as produced by
notebooks 01-09. The unnumbered cell near the top is an ad-hoc filesystem scan used while
locating the prediction files; it is not part of the verification and is not required.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, numpy as np
RES = '/content/drive/MyDrive/ADR_Project/results'
print([f for f in sorted(os.listdir(RES)) if f.endswith('.npz')])

[]


In [3]:
print(sorted(os.listdir(RES)))

['ablation_results.csv', 'atom_attribution_metformin.csv', 'baseline_xgboost_metrics.csv', 'bootstrap_test.csv', 'disproportionality_results.csv', 'figs_data', 'fuzzy_match_audit.csv', 'gnn_test_metrics.csv', 'high_risk_priority_list.csv', 'interpretability_summary.csv', 'multimodal_test_metrics.csv', 'pair_overlap_audit.csv', 'pair_overlap_summary.csv', 'score_distribution.png', 'shap_adr_dimensions.csv', 'shap_adr_summary.csv', 'sider_per_drug_exact.csv', 'sider_sensitivity.csv', 'sider_validation.csv', 'sider_validation_pairs.csv', 'sider_validation_per_drug.csv']


In [5]:
import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

for f in ['gnn_test_metrics.csv', 'multimodal_test_metrics.csv',
          'baseline_xgboost_metrics.csv', 'ablation_results.csv']:
    print(f'===== {f}')
    print(pd.read_csv(f'{RES}/{f}'))
    print()

===== gnn_test_metrics.csv
      AUROC     AUPRC  Sensitivity  Precision
0  0.824249  0.839325     0.727152   0.789689

===== multimodal_test_metrics.csv
      AUROC     AUPRC  Sensitivity  Precision  Accuracy
0  0.850903  0.857594     0.734694   0.810573    0.7815

===== baseline_xgboost_metrics.csv
      AUROC     AUPRC  Accuracy  Sensitivity  Precision        F1
0  0.741375  0.743666  0.734361     0.522626   0.906503  0.663008

===== ablation_results.csv
               model     AUROC     AUPRC  Sensitivity  Precision  Accuracy
0           GNN only  0.824249       NaN          NaN        NaN       NaN
1      Concat fusion  0.851206  0.858070     0.716948   0.820305  0.779947
2  Cross-attn fusion  0.850903  0.857594     0.734694   0.810573  0.781500



In [6]:
FIGS = f'{RES}/figs_data'
print(sorted(os.listdir(FIGS)))

d = np.load(f'{FIGS}/crossattn_preds.npz', allow_pickle=True)
print('keys:', d.files)
for k in d.files:
    a = np.asarray(d[k])
    print(f'  {k}: shape={a.shape} dtype={a.dtype} min={a.min() if a.dtype.kind in "fi" else "-"} max={a.max() if a.dtype.kind in "fi" else "-"}')

['crossattn_preds.npz', 'fig_atom_attribution.pdf', 'operating_points.csv', 'per_class_auroc.csv', 'table54_summary.csv', 'test_prr_join.csv']
keys: ['drug', 'reaction', 'y_true', 'crossattn']
  drug: shape=(9016,) dtype=object min=- max=-
  reaction: shape=(9016,) dtype=object min=- max=-
  y_true: shape=(9016,) dtype=int64 min=0 max=1
  crossattn: shape=(9016,) dtype=float32 min=0.018904132768511772 max=0.992171049118042


In [7]:
print('===== shap_adr_summary.csv')
print(pd.read_csv(f'{RES}/shap_adr_summary.csv'))
print()
print('===== shap_adr_dimensions.csv (top 10)')
sd = pd.read_csv(f'{RES}/shap_adr_dimensions.csv')
print(sd.head(10))
print('rows:', len(sd), '| columns:', sd.columns.tolist())

===== shap_adr_summary.csv
        drug  n_background  n_explained  n_dimensions  nsamples  total_mean_abs_shap  max_mean_abs_shap  share_top1  share_top20  share_top100  n_dims_for_half_attribution  explained_score_min  \
0  metformin            50           50           768      2000             0.085008            0.00279    0.032823     0.369211      0.808401                           34             0.624645   

   explained_score_max  explained_score_range  
0             0.982106                0.35746  

===== shap_adr_dimensions.csv (top 10)
   dimension  mean_abs_shap  rank
0        767       0.002790     1
1        572       0.002584     2
2        441       0.002203     3
3        261       0.002044     4
4        365       0.002002     5
5        642       0.001775     6
6        562       0.001747     7
7        493       0.001659     8
8         59       0.001588     9
9        337       0.001496    10
rows: 768 | columns: ['dimension', 'mean_abs_shap', 'rank']


In [8]:
n, pos, neg = 9016, 4508, 4508

def derive_accuracy(sens, prec, name):
    TP   = round(sens * pos)
    FP   = round(TP / prec - TP)
    TN   = neg - FP
    acc  = (TP + TN) / n
    print(f'{name:12s} TP={TP:5d} FP={FP:4d} TN={TN:5d} '
          f'pred_pos={TP+FP:5d}  ACC={acc:.4f}')

derive_accuracy(0.727152, 0.789689, 'GAT')        # -> 0.7667  (target value)
derive_accuracy(0.734694, 0.810573, 'Cross-attn') # -> 0.7815  matches logged file
derive_accuracy(0.716948, 0.820305, 'Concat')     # -> 0.7799  matches logged file
derive_accuracy(0.522626, 0.906503, 'XGBoost')    # -> 0.7344  matches logged file

GAT          TP= 3278 FP= 873 TN= 3635 pred_pos= 4151  ACC=0.7667
Cross-attn   TP= 3312 FP= 774 TN= 3734 pred_pos= 4086  ACC=0.7815
Concat       TP= 3232 FP= 708 TN= 3800 pred_pos= 3940  ACC=0.7799
XGBoost      TP= 2356 FP= 243 TN= 4265 pred_pos= 2599  ACC=0.7344


In [9]:
d = np.load(f'{FIGS}/crossattn_preds.npz', allow_pickle=True)
y, p = d['y_true'], d['crossattn']

print(f'pos mean = {p[y==1].mean():.4f}   median = {np.median(p[y==1]):.4f}')
print(f'neg mean = {p[y==0].mean():.4f}   median = {np.median(p[y==0]):.4f}')
print(f'separation = {p[y==1].mean() - p[y==0].mean():.4f}')
print(f'pred positive at 0.5 = {(p>=0.5).sum()} / {len(p)}')
print()
print(pd.read_csv(f'{RES}/interpretability_summary.csv').T)

pos mean = 0.6541   median = 0.7000
neg mean = 0.3313   median = 0.3083
separation = 0.3228
pred positive at 0.5 = 4086 / 9016

                                                                                                 0
attention_analysis                               Cross-attention weights show atom-level releva...
attention_correlation_lactic_acidosis_vs_nausea                                           0.996271
shap_n_dimensions                                                                              768
shap_share_top20                                                                          0.369211
shap_dims_for_half_attribution                                                                  34
occlusion_max_effect                                                                      0.087029
pos_score_mean                                                                            0.847452
neg_score_mean                                                                  

In [10]:
drugs = np.array([str(x).lower() for x in d['drug']])
m = drugs == 'metformin'
print(f'metformin pairs: n={m.sum()}  positives={int(y[m].sum())}')
print(f'  pos mean = {p[m & (y==1)].mean():.4f}')
print(f'  neg mean = {p[m & (y==0)].mean():.4f}')
print(f'  gap      = {p[m & (y==1)].mean() - p[m & (y==0)].mean():.4f}')
print(f'  score range = {p[m].min():.4f} to {p[m].max():.4f}')

metformin pairs: n=2052  positives=1889
  pos mean = 0.8475
  neg mean = 0.8099
  gap      = 0.0376
  score range = 0.1971 to 0.9922


In [11]:
from sklearn.metrics import roc_auc_score

rows = []
for dr in sorted(set(drugs)):
    mk = drugs == dr
    yy, pp = y[mk], p[mk]
    auc = roc_auc_score(yy, pp) if 0 < yy.sum() < len(yy) else np.nan
    rows.append({'drug': dr, 'n': mk.sum(), 'pos_rate': round(yy.mean(), 3),
                 'within_drug_auroc': round(auc, 3) if auc == auc else None,
                 'score_mean': round(pp.mean(), 3)})

wd = pd.DataFrame(rows).sort_values('n', ascending=False)
print(wd.to_string(index=False))
print()
print(f"overall AUROC          = {roc_auc_score(y, p):.3f}")
print(f"mean within-drug AUROC = {wd['within_drug_auroc'].mean():.3f}")
wd.to_csv(f'{RES}/within_drug_auroc.csv', index=False)

         drug    n  pos_rate  within_drug_auroc  score_mean
    metformin 2052     0.921              0.617       0.844
dapagliflozin 1173     0.783              0.737       0.503
empagliflozin  757     0.598              0.775       0.492
  linagliptin  676     0.512              0.712       0.189
canagliflozin  535     0.327              0.780       0.485
    glipizide  504     0.325              0.865       0.446
  sitagliptin  494     0.332              0.758       0.405
 pioglitazone  465     0.275              0.786       0.429
  glimepiride  459     0.307              0.809       0.485
    glyburide  386     0.111              0.863       0.350
  repaglinide  384     0.115              0.862       0.319
  saxagliptin  381     0.068              0.940       0.264
     acarbose  377     0.032              0.894       0.191
  nateglinide  373     0.011              0.977       0.222

overall AUROC          = 0.851
mean within-drug AUROC = 0.812


In [12]:
from scipy.stats import pearsonr, spearmanr
w = wd.dropna(subset=['within_drug_auroc'])
print(f"weighted mean AUROC = {(w.within_drug_auroc * w.n).sum() / w.n.sum():.3f}")
print(f"pos_rate vs AUROC     spearman = {spearmanr(w.pos_rate, w.within_drug_auroc)[0]:.3f}")
print(f"log(n) vs score_mean  pearson  = {pearsonr(np.log(w.n), w.score_mean)[0]:.3f}")

weighted mean AUROC = 0.764
pos_rate vs AUROC     spearman = -0.925
log(n) vs score_mean  pearson  = 0.782


In [13]:
print(len(set(drugs)))
print(sorted(set(drugs)))

14
[np.str_('acarbose'), np.str_('canagliflozin'), np.str_('dapagliflozin'), np.str_('empagliflozin'), np.str_('glimepiride'), np.str_('glipizide'), np.str_('glyburide'), np.str_('linagliptin'), np.str_('metformin'), np.str_('nateglinide'), np.str_('pioglitazone'), np.str_('repaglinide'), np.str_('saxagliptin'), np.str_('sitagliptin')]


In [14]:
import glob
for f in glob.glob('/content/drive/MyDrive/ADR_Project/data/**/*.csv', recursive=True):
    if any(k in f.lower() for k in ['drug', 'smiles', 'cohort']):
        print(f)

/content/drive/MyDrive/ADR_Project/data/raw/antidiabetic_drugs_smiles.csv
/content/drive/MyDrive/ADR_Project/data/processed/drug_adr_with_structures.csv


In [15]:
cohort = pd.read_csv('/content/drive/MyDrive/ADR_Project/data/raw/antidiabetic_drugs_smiles.csv')
print(cohort.columns.tolist())
print(f'cohort size = {len(cohort)}')

# Locate the drug-name column (usually 'drug', 'drug_name' or 'name')
col = [c for c in cohort.columns if 'drug' in c.lower() or 'name' in c.lower()][0]
cohort_names = set(cohort[col].astype(str).str.lower().str.strip())
test_names   = set(str(x).lower() for x in drugs)

print(f'\nin cohort but NOT in test split: {sorted(cohort_names - test_names)}')
print(f'in test but not in cohort      : {sorted(test_names - cohort_names)}')

['drug_name', 'pubchem_cid', 'smiles', 'formula', 'mol_weight']
cohort size = 15

in cohort but NOT in test split: ['rosiglitazone']
in test but not in cohort      : []


In [16]:
full = pd.read_csv('/content/drive/MyDrive/ADR_Project/data/processed/drug_adr_with_structures.csv')
c2 = [c for c in full.columns if 'drug' in c.lower()][0]
names = full[c2].astype(str).str.lower().str.strip()
print(f'drugs in processed data = {names.nunique()}')
print(f"rosiglitazone rows = {(names == 'rosiglitazone').sum()}")

drugs in processed data = 15
rosiglitazone rows = 7


In [17]:
t = pd.read_csv(f'{FIGS}/test_prr_join.csv')
print(t.columns.tolist())
print(len(t))
print(t.head())

['drug', 'pubchem_cid', 'smiles', 'adr', 'label', 'k', 'a', 'prr', 'in_dis', 'scoreable', 'prr_signal']
9016
            drug  pubchem_cid                                             smiles                        adr  label                                       k     a        prr  in_dis  scoreable  \
0      metformin         4091                                  CN(C)C(=N)N=C(N)N           Skin haemorrhage      1             metformin||skin haemorrhage  16.0   1.544833    True       True   
1  dapagliflozin      9887712  CCOC1=CC=C(C=C1)CC2=C(C=CC(=C2)C3C(C(C(C(O3)CO...          Dermatitis diaper      1        dapagliflozin||dermatitis diaper   1.0  98.660450    True      False   
2  dapagliflozin      9887712  CCOC1=CC=C(C=C1)CC2=C(C=CC(=C2)C3C(C(C(C(O3)CO...            Nephrolithiasis      1          dapagliflozin||nephrolithiasis   2.0   2.610065    True      False   
3    sitagliptin      4369359  C1CN2C(=NN=C2C(F)(F)F)CN1C(=O)CC(CC3=CC(=C(C=C...          Suspected suicide      0 

In [18]:
t['a_filled'] = t['a'].fillna(0)

print('=== fraction of pairs with a > 0, by label ===')
print(t.groupby('label')['a_filled'].apply(lambda s: (s > 0).mean()).round(3))
print()
print('=== mean joint count a, by label ===')
print(t.groupby('label')['a_filled'].mean().round(2))
print()
print(f"label=1 with a=0 : {((t.label==1) & (t.a_filled==0)).sum()}")
print(f"label=0 with a>0 : {((t.label==0) & (t.a_filled>0)).sum()}")

=== fraction of pairs with a > 0, by label ===
label
0    0.054
1    0.523
Name: a_filled, dtype: float64

=== mean joint count a, by label ===
label
0    0.11
1    9.53
Name: a_filled, dtype: float64

label=1 with a=0 : 2152
label=0 with a>0 : 243


In [19]:
from sklearn.metrics import roc_auc_score

# 1) Confirm that t has the same row order as the saved npz predictions
d = np.load(f'{FIGS}/crossattn_preds.npz', allow_pickle=True)
same_order = (np.array([str(x).lower() for x in d['drug']]) ==
              t['drug'].astype(str).str.lower().values).all()
print(f'same row order as npz: {same_order}')
print(f"label matches y_true : {(t['label'].values == d['y_true']).all()}")

# 2) Count-only AUROC over the full test set
print(f"\ncount-only AUROC = {roc_auc_score(t.label, t.a_filled):.3f}")
print(f"fusion AUROC     = {roc_auc_score(d['y_true'], d['crossattn']):.3f}")

# 3) Confirm the XGBoost baseline is equivalent to the rule 'a > 0'
from sklearn.metrics import accuracy_score, precision_score, recall_score
yhat = (t.a_filled > 0).astype(int)
print(f"\nrule 'a>0': acc={accuracy_score(t.label, yhat):.3f} "
      f"prec={precision_score(t.label, yhat):.3f} "
      f"rec={recall_score(t.label, yhat):.3f}")
print("XGBoost   : acc=0.734 prec=0.907 rec=0.523")

# 4) Derive the observed / novel pair flag
for f in ['pair_overlap_audit.csv', 'pair_overlap_summary.csv']:
    print(f'\n===== {f}')
    x = pd.read_csv(f'{RES}/{f}')
    print(x.shape, x.columns.tolist())
    print(x.head())

same row order as npz: True
label matches y_true : True

count-only AUROC = 0.741
fusion AUROC     = 0.851

rule 'a>0': acc=0.734 prec=0.907 rec=0.523
XGBoost   : acc=0.734 prec=0.907 rec=0.523

===== pair_overlap_audit.csv
(3, 4) ['subset', 'n', 'positive_rate', 'auroc']
                      subset     n  positive_rate   auroc
0  Previously observed pairs  2204         0.9106  0.6918
1                Novel pairs  6812         0.3671  0.7816
2             All test pairs  9016         0.5000  0.8509

===== pair_overlap_summary.csv
(3, 4) ['subset', 'n', 'positive_rate', 'AUROC']
                subset     n  positive_rate   AUROC
0  Previously observed  2204         0.9106  0.6918
1           Novel pair  6812         0.3671  0.7816
2       All test pairs  9016         0.5000  0.8509


In [20]:
# Locate the training split file
import glob
for f in glob.glob('/content/drive/MyDrive/ADR_Project/data/**/*.csv', recursive=True):
    if any(k in f.lower() for k in ['train', 'split', 'pair']):
        print(f)

/content/drive/MyDrive/ADR_Project/data/processed/train.csv


In [22]:
TRAIN = '/content/drive/MyDrive/ADR_Project/data/processed/train.csv'
tr = pd.read_csv(TRAIN)
print(tr.columns.tolist())
print(f'train rows = {len(tr)}')
print(tr.head(3))

['drug', 'pubchem_cid', 'smiles', 'adr', 'label']
train rows = 8240
          drug  pubchem_cid                                             smiles                           adr  label
0    glipizide         3478  CC1=CN=C(C=N1)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC...                  Constipation      1
1  nateglinide      5311309         CC(C)C1CCC(CC1)C(=O)NC(CC2=CC=CC=C2)C(=O)O  Hyperplastic cholecystopathy      0
2    metformin         4091                                  CN(C)C(=N)N=C(N)N         Hepatitis cholestatic      1


In [23]:
norm = lambda s: s.astype(str).str.lower().str.strip()

train_pos = set(zip(norm(tr.loc[tr.label == 1, 'drug']),
                    norm(tr.loc[tr.label == 1, 'adr'])))
print(f'train positives (rows)   = {(tr.label==1).sum()}')
print(f'train positives (unique) = {len(train_pos)}')   # expected: 4,120

train positives (rows)   = 4120
train positives (unique) = 4120


In [24]:
from sklearn.metrics import roc_auc_score

t['observed'] = [(dd, aa) in train_pos
                 for dd, aa in zip(norm(t['drug']), norm(t['adr']))]
t['fusion'] = d['crossattn']

rows = [('Previously observed pairs', t[t.observed]),
        ('Novel pairs',               t[~t.observed]),
        ('All test pairs',            t)]

res = pd.DataFrame([{
    'subset': nm, 'n': len(s),
    'positive_rate':    round(s.label.mean(), 4),
    'fusion_auroc':     round(roc_auc_score(s.label, s.fusion), 4),
    'count_only_auroc': round(roc_auc_score(s.label, s.a_filled), 4),
} for nm, s in rows])

print()
print(res.to_string(index=False))
print()
print('--- expected values (Table 4.2) ---')
print('n              2204 / 6812 / 9016')
print('positive_rate  0.9106 / 0.3671 / 0.5000')
print('fusion_auroc   0.6918 / 0.7816 / 0.8509')


                   subset    n  positive_rate  fusion_auroc  count_only_auroc
Previously observed pairs 2204         0.9106        0.6918            0.7876
              Novel pairs 6812         0.3671        0.7816            0.5645
           All test pairs 9016         0.5000        0.8509            0.7413

--- expected values (Table 4.2) ---
n              2204 / 6812 / 9016
positive_rate  0.9106 / 0.3671 / 0.5000
fusion_auroc   0.6918 / 0.7816 / 0.8509


In [25]:
res.to_csv(f'{RES}/pair_overlap_audit.csv', index=False)
print(pd.read_csv(f'{RES}/pair_overlap_audit.csv'))

                      subset     n  positive_rate  fusion_auroc  count_only_auroc
0  Previously observed pairs  2204         0.9106        0.6918            0.7876
1                Novel pairs  6812         0.3671        0.7816            0.5645
2             All test pairs  9016         0.5000        0.8509            0.7413


In [26]:
for f in ['sider_validation.csv', 'sider_sensitivity.csv',
          'sider_per_drug_exact.csv', 'fuzzy_match_audit.csv']:
    print(f'===== {f}')
    x = pd.read_csv(f'{RES}/{f}')
    print(x.shape, x.columns.tolist())
    print(x.head(3))
    print()

===== sider_validation.csv
(1, 10) ['n_drugs_total', 'n_drugs_in_sider', 'n_positive_pairs', 'n_mappable_pairs', 'n_confirmed_by_sider', 'precision_vs_sider', 'recall_on_sider', 'coverage', 'n_vocab_overlap', 'n_fuzzy_matches']
   n_drugs_total  n_drugs_in_sider  n_positive_pairs  n_mappable_pairs  n_confirmed_by_sider  precision_vs_sider  recall_on_sider  coverage  n_vocab_overlap  n_fuzzy_matches
0             15                13              6621              1686                   539            0.319692         0.569767  0.254644              328               99

===== sider_sensitivity.csv
(2, 7) ['Unnamed: 0', 'mappable', 'confirmed', 'precision', 'recall', 'coverage', 'sider_denominator']
    Unnamed: 0  mappable  confirmed  precision  recall  coverage  sider_denominator
0   exact_only    1467.0      474.0     0.3231  0.5043    0.2216              940.0
1  exact_fuzzy    1686.0      539.0     0.3197  0.5698    0.2546              946.0

===== sider_per_drug_exact.csv
(13, 5) 